In [15]:
import os
import glob
import pandas as pd
import numpy as np
import kagglehub

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression

In [16]:
# STEP 1: DOWNLOAD AND ACCESS THE DATASET
# =====================================================================
print("📥 Fetching the base handheld device dataset from Kaggle...")
path = kagglehub.dataset_download("ahsan81/used-handheld-device-data")

csv_files = glob.glob(os.path.join(path, "*.csv"))
if not csv_files:
    raise FileNotFoundError("Could not locate any valid CSV spreadsheet.")

raw_df = pd.read_csv(csv_files[0])

📥 Fetching the base handheld device dataset from Kaggle...


In [17]:
# STEP 2: COMPLETE BRAND LIST INTEGRATION & DATA ENGINEERING
# =====================================================================
# The exact list of 30 target companies requested
TARGET_BRANDS = [
    'Apple', 'Samsung', 'Xiaomi', 'Oppo', 'Vivo', 'Huawei', 'Honor', 
    'Motorola', 'Google Pixel', 'Oneplus', 'Nothing', 'Sony Xperia', 
    'Nokia', 'Realme', 'Asus', 'Lenovo', 'Zte', 'Tcl', 'Infinix', 
    'Tecno', 'Itel', 'Sparx', 'Qmobile', 'Dcode', 'Htc', 'Lg', 
    'Micromax', 'Lava', 'Karbonn', 'Blackberry'
]

# Standardize raw dataset brand names
selected_columns = ['device_brand', 'internal_memory', 'days_used', 'normalized_used_price']
df = raw_df[selected_columns].copy().dropna()
df['device_brand'] = df['device_brand'].str.strip().str.title()

# Map common user text inputs to standard names
brand_synonyms = {
    'iphone': 'Apple', 'redmi': 'Xiaomi', 'redme': 'Xiaomi', 
    'google': 'Google Pixel', 'pixel': 'Google Pixel',
    'oppo': 'Oppo', 'oneplus': 'Oneplus', 'sony': 'Sony Xperia',
    'itel': 'Itel', 'qmobile': 'Qmobile', 'htc': 'Htc', 'lg': 'Lg'
}

# Engineer a 70-100% Battery Health feature from usage age
np.random.seed(42)
df['battery_health'] = 100 - (df['days_used'] / 45) + np.random.normal(0, 1.5, len(df))
df['battery_health'] = df['battery_health'].clip(70, 100).round(1)
df = df.drop(columns=['days_used'])

# Real-world market tier definitions to seed missing local/legacy brands
tier_profiles = {
    'premium': ['Apple', 'Google Pixel', 'Samsung', 'Oneplus', 'Nothing', 'Sony Xperia', 'Asus', 'Blackberry'],
    'mid': ['Xiaomi', 'Oppo', 'Vivo', 'Huawei', 'Honor', 'Motorola', 'Realme', 'Zte', 'Htc', 'Lg', 'Lenovo'],
    'budget': ['Infinix', 'Tecno', 'Itel', 'Sparx', 'Qmobile', 'Dcode', 'Nokia', 'Tcl', 'Micromax', 'Lava', 'Karbonn']
}

# Data Infill: Inject realistic pricing rows if a brand isn't in the Kaggle file
print("🛠️ Syncing missing brand matrix data...")
existing_brands = df['device_brand'].unique()
synthetic_rows = []

for brand in TARGET_BRANDS:
    if brand not in existing_brands:
        # Determine baseline target pricing score based on market category tier
        if brand in tier_profiles['premium']:
            base_score = 5.8  # High value flagship scale
        elif brand in tier_profiles['mid']:
            base_score = 4.8  # Balanced mid-tier scale
        else:
            base_score = 4.0  # Affordable mass market budget scale
            
        # Generate 30 distinct specification variations so the linear model learns the trend
        for i in range(30):
            mem = np.random.choice([32, 64, 128, 256])
            health = np.random.uniform(75, 100)
            # Memory adjustments (+ value for storage) and health degradation logic
            adjusted_score = base_score + (mem / 512) + ((health - 80) / 100)
            synthetic_rows.append({
                'device_brand': brand,
                'internal_memory': float(mem),
                'battery_health': round(health, 1),
                'normalized_used_price': adjusted_score
            })

if synthetic_rows:
    df = pd.concat([df, pd.DataFrame(synthetic_rows)], ignore_index=True)

🛠️ Syncing missing brand matrix data...


In [18]:
# STEP 3: ONE-HOT ENCODING & SCALING
# =====================================================================
X_raw = df[['device_brand', 'internal_memory', 'battery_health']]
y = df['normalized_used_price']

# Perform One-Hot Encoding across the synchronized dataset
X_encoded = pd.get_dummies(X_raw, columns=['device_brand'], dtype=float)
feature_cols = X_encoded.columns.tolist()

numeric_cols = ['internal_memory', 'battery_health']
scaler = StandardScaler()

X_scaled = X_encoded.copy()
X_scaled[numeric_cols] = scaler.fit_transform(X_encoded[numeric_cols])

# Train the final Multiple Linear Regression Model
model = LinearRegression()
model.fit(X_scaled, y)
print("✅ Fully loaded brand regression engine complete.")

✅ Fully loaded brand regression engine complete.


In [ ]:
# STEP 4: INTERACTIVE RUNTIME PREDICTIONS INTERFACE
# =====================================================================
print("\n==================================================================")
print("🎯 WELCOME TO THE COMPLETE MOBILE PRICING & RATING SYSTEM 🎯")
print("==================================================================")

while True:
    print("\nEnter specifications for the phone you wish to value (or type 'exit' to quit):")
    
    try:
        # Exact prompt structure matching user instructions
        user_brand = input("👉 Enter mobile phone name: ")
        if user_brand.lower() == 'exit':
            break
            
        user_storage = input("👉 Storage capacity in GB: ")
        user_health = input("👉 Enter Battery Health Percentage(0-100): ")
        
        # Format and validate the brand entry text
        brand_clean = user_brand.strip().lower()
        mapped_brand = brand_synonyms.get(brand_clean, user_brand.strip().title())
        
        storage = float(user_storage)
        health = float(user_health)
        
        if health < 0 or health > 100:
            print("❌ Input Error: Battery health percentage must fall between 0 and 100.")
            continue
            
        # Initialize inference row with 0s matching training shape
        input_row = pd.DataFrame(0.0, index=[0], columns=feature_cols)
        input_row['internal_memory'] = storage
        input_row['battery_health'] = health
        
        # Activate corresponding One-Hot encoded brand column flag
        target_dummy_col = f"device_brand_{mapped_brand}"
        if target_dummy_col in feature_cols:
            input_row[target_dummy_col] = 1.0
        else:
            # Fallback handling for entirely unlisted brands
            closest_fallback = 'Samsung'
            if mapped_brand.lower() in [b.lower() for b in tier_profiles['budget']]:
                closest_fallback = 'Infinix'
            elif mapped_brand.lower() in [b.lower() for b in tier_profiles['premium']]:
                closest_fallback = 'Apple'
            input_row[f"device_brand_{closest_fallback}"] = 1.0
            
        # Scale continuous metrics
        input_row_scaled = input_row.copy()
        input_row_scaled[numeric_cols] = scaler.transform(input_row[numeric_cols])
        
        # Calculate raw regression model output score
        predicted_score = model.predict(input_row_scaled)[0]
        
        # -----------------------------------------------------------------
        # COMPREHENSIVE CURRENCY MAPPING & DYNAMIC STAR DISTRIBUTION
        # -----------------------------------------------------------------
        min_possible_score, max_possible_score = 3.5, 6.5
        clipped_score = max(min_possible_score, min(max_possible_score, predicted_score))
        
        # Map score to standard global USD value bounds ($40 - $1400)
        min_usd, max_usd = 40, 1400
        usd_price = min_usd + ((clipped_score - min_possible_score) / (max_possible_score - min_possible_score)) * (max_usd - min_usd)
        
        # Convert to PKR (Using updated baseline exchange standard: 1 USD = 280 PKR)
        pkr_price = usd_price * 280
        
        # Updated Star Adjustment Logic (Increases/decreases directly with valuation tier)
        if usd_price < 120:
            star_count = 1
            device_tier = "Budget Tier"
        elif usd_price < 300:
            star_count = 2
            device_tier = "Entry Mid-Range"
        elif usd_price < 600:
            star_count = 3
            device_tier = "Mid-Range"
        elif usd_price < 950:
            star_count = 4
            device_tier = "Upper Mid-Range"
        else:
            star_count = 5
            device_tier = "Premium Flagship"
            
        star_rating_visual = "⭐" * star_count
        
        # Output Terminal Display Panel
        print("\n" + "💰" * 20)
        print(f"🧠 REGRESSION ANALYSIS FOR: {mapped_brand.upper()}")
        print("" + "━" * 38)
        print(f"💵 Price in USD: ${usd_price:,.2f}")
        print(f"🇵🇰 Price in PKR: Rs. {pkr_price:,.2f}")
        print(f"📊 Device Class: {device_tier}")
        print(f"🏅 Premium Tier Rating: {star_rating_visual} ({star_count}/5 Stars)")
        print("💰" * 20)
        
    except ValueError:
        print("\n❌ Input Error: Please enter numerical characters only for storage and battery health fields.")
    except Exception as e:
        print(f"\n❌ Pipeline runtime exception encountered: {e}")


🎯 WELCOME TO THE COMPLETE MOBILE PRICING & RATING SYSTEM 🎯

Enter specifications for the phone you wish to value (or type 'exit' to quit):


👉 Enter mobile phone name:  Iphone
👉 Storage capacity in GB:  128
👉 Enter Battery Health Percentage(0-100):  96



💰💰💰💰💰💰💰💰💰💰💰💰💰💰💰💰💰💰💰💰
🧠 REGRESSION ANALYSIS FOR: APPLE
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
💵 Price in USD: $829.13
🇵🇰 Price in PKR: Rs. 232,155.51
📊 Device Class: Upper Mid-Range
🏅 Premium Tier Rating: ⭐⭐⭐⭐ (4/5 Stars)
💰💰💰💰💰💰💰💰💰💰💰💰💰💰💰💰💰💰💰💰

Enter specifications for the phone you wish to value (or type 'exit' to quit):


👉 Enter mobile phone name:  Huawei
👉 Storage capacity in GB:  128
👉 Enter Battery Health Percentage(0-100):  81



💰💰💰💰💰💰💰💰💰💰💰💰💰💰💰💰💰💰💰💰
🧠 REGRESSION ANALYSIS FOR: HUAWEI
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
💵 Price in USD: $522.52
🇵🇰 Price in PKR: Rs. 146,306.83
📊 Device Class: Mid-Range
🏅 Premium Tier Rating: ⭐⭐⭐ (3/5 Stars)
💰💰💰💰💰💰💰💰💰💰💰💰💰💰💰💰💰💰💰💰

Enter specifications for the phone you wish to value (or type 'exit' to quit):
